## Preparation

First, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure everyone works with the exact same data.

We will use gitsource for that:

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. Because we specify allowed_extensions={"md"}, it only checks the markdown files.

We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

Each file has a parse() method that returns a dictionary with its filename and content:

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

How many lesson pages are in the dataset?

In [ ]:
len(documents)

## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [ ]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results

## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.


Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

In [ ]:
import os 
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client,
)

answer, tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(tokens)


## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
len(chunks)

## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?


In [ ]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [ ]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client,
)

answer, tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(tokens)

## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.


Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

In [ ]:
instructions = '''
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.'''

In [ ]:
from langchain_core.tools import tool

@tool
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google-genai",
    temperature=0.5,
    timeout=600,
    max_tokens=25000,
    streaming=True,
)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[search],
    system_prompt=instructions,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}]}
)
print(result["messages"][-1].text)

history = result.get("intermediate_steps", [])

# Since 'intermediate_steps' tracks every action taken:
total_calls = len(history)
print(total_calls)